In [7]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import time

In [2]:
#stat1 = pd.read_csv("transactions_verbs_obl_kohakaandes_patterns_stats_v1.csv",sep=",", encoding="utf-8")
stat2 = pd.read_csv("transactions_verbs_obl_kohakaandes_patterns_stats_v2.csv",sep=",", encoding="utf-8")

In [3]:
stat2 = stat2.sort_values('pat_count', ascending=False)

In [55]:
stat2_2 = stat2.drop(["pat_count"], axis=1)

In [56]:
stat2_2

,root,opteerima,korrigeerima,affima,puhkama,eitama,saaxima,saukima,haugatama,trahvima,...,propageerima,jääma,voolama,pilutama,teisendama,krissuma,töusnuma,käima,ammuma,meilima
50953,aeg,0,0,0,1,1,0,0,0,1,...,1,1,1,0,0,0,0,1,0,0
21500,aasta,1,1,0,1,1,0,0,0,1,...,1,1,1,0,0,0,0,1,0,1
30192,sõna,0,1,0,1,1,0,0,0,1,...,1,1,1,0,0,0,0,1,0,0
51095,mina,0,0,0,1,1,1,0,0,0,...,1,1,1,0,0,0,0,1,0,1
17567,tema,0,1,0,1,1,0,0,0,1,...,0,1,1,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15767,tööhöiveamet,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
35150,TAAS,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
35149,traadipea,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
35148,kenadus,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [4]:
stat2

,root,opteerima,korrigeerima,affima,puhkama,eitama,saaxima,saukima,haugatama,trahvima,...,jääma,voolama,pilutama,teisendama,krissuma,töusnuma,käima,ammuma,meilima,pat_count
50953,aeg,0,0,0,1,1,0,0,0,1,...,1,1,0,0,0,0,1,0,0,1697
21500,aasta,1,1,0,1,1,0,0,0,1,...,1,1,0,0,0,0,1,0,1,1686
30192,sõna,0,1,0,1,1,0,0,0,1,...,1,1,0,0,0,0,1,0,0,1561
51095,mina,0,0,0,1,1,1,0,0,0,...,1,1,0,0,0,0,1,0,1,1547
17567,tema,0,1,0,1,1,0,0,0,1,...,1,1,0,0,0,0,1,0,1,1426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15767,tööhöiveamet,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
35150,TAAS,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
35149,traadipea,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
35148,kenadus,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1


In [ ]:
# collection (S): set(root+selle alla kuuluvad mustrid) * number_of_roots
# m: setide arv collectionis
# union of S = universe

In [5]:
mustrid = list(stat2.columns)[1:-1]  # universe

In [6]:
roots = list(stat2["root"])

In [66]:
subsets1 = {}

for i in tqdm(range(len(stat2_2))):
    root = stat2_2.iloc[i]["root"]
    row = stat2_2[stat2_2["root"]==root]
    col = (row == 1).any()
    d = dict(col)
    pats = []
    for k in d.keys():
        if d[k]==True:
            pats.append(k)
    subsets1[root] = pats


100%|█████████████████████████████████████| 55103/55103 [36:40<00:00, 25.04it/s]


In [156]:
len(subsets1)

55103

In [86]:
# numbrite asemel on verbid, siis määrata igale verbile number

verb_to_num = {}
num_to_verb = {}

for i,e in enumerate(mustrid):
    verb_to_num[e] = i+1
    num_to_verb[i+1] = e


In [87]:
# teha subsets2 kus verbide asemel on igas setis numbrid

subsets2 = {}

for k in tqdm(subsets1.keys()):
    values = subsets1[k]
    values2 = [verb_to_num[v] for v in values]
    subsets2[k] = values2


100%|█████████████████████████████████| 55103/55103 [00:00<00:00, 303199.98it/s]


In [73]:
#keys = list(subsets2.keys())[:3]
#subsets2[keys[0]]

Set cover koodi alus

https://github.com/AndreaRubbi/Set-Cover-problem-solution-Python/blob/master/SetCover.pdf

https://github.com/AndreaRubbi/Set-Cover-problem-solution-Python/blob/master/Greedy.py

__author__ = "Andrea Rubbi"


In [150]:
def set_cover(universe, subsets,costs):
    cost=0
    elements = set(e for s in subsets for e in s)
    
    sorted_elements = sorted(elements)
    sorted_universe = sorted(universe)
    
    print(len(elements), len(universe), sorted_elements==sorted_universe)
    
    if sorted_elements != sorted_universe:
        return None
    covered = set()
    cover = []
    sorted_covered = sorted(covered)
    
    #print(len(sorted_covered), )
    
    while sorted_covered != sorted_elements:
        subset = max(subsets, key=lambda s: len(s - covered)/costs[subsets.index(s)])
        cover.append(subset)
        cost+=costs[subsets.index(subset)]
        covered |= subset
        sorted_covered = sorted(covered)
        
        print(len(sorted_covered), len(sorted_elements))
 
    return cover, cost
 

In [138]:
def main(a,b,c,d, x=time.time()):
    m= a
    universe = d
    sub = b  
    
    subsets = [set(x) for x in sub]
    costs =  c 
    cover = set_cover(universe, subsets,costs)
    #print('covering sets= ',cover[0],'\n',
    #      'cost= ',cover[1],'$')
    print('time: ',time.time()-x)
    
    return cover[0]

### testida 10 setiga aega

In [131]:
S1 = [subsets2[k] for k in subsets2.keys()][:10]
uni = list(set([e for s in S1 for e in s]))
m1 = max(uni)

P1 = [1]*len(S1) # anda igale setile cost(weight) 1


In [139]:
%%time

if __name__ == '__main__':
    covering_sets = main(m1,S1,P1,uni)

2914 2914 True
time:  3.024367332458496
CPU times: user 13.8 ms, sys: 0 ns, total: 13.8 ms
Wall time: 13.2 ms


In [141]:
len(covering_sets)

10

In [144]:
covering_roots = []

for cov_set in covering_sets:
    for k in subsets2.keys():
        subset = sorted(subsets2[k])
        if subset == sorted(cov_set):
            covering_roots.append(k)
        

In [145]:
covering_roots

['aeg',
 'mina',
 'aasta',
 'sõna',
 'tema',
 'see',
 'mis',
 'päev',
 'algus',
 'Eesti']

### kõik originaal setid

In [151]:
#m1 = max(num_to_verb.keys())
#S1 = [subsets1[k] for k in subsets1.keys()]
#P1 = [1]*len(S1) # anda igale setile cost(weight) 1
#uni = list(verb_to_num.keys())

S1 = [subsets2[k] for k in subsets2.keys()]
uni = list(set([e for s in S1 for e in s]))
m1 = max(uni)

P1 = [1]*len(S1) # anda igale setile cost(weight) 1


In [152]:
%%time

if __name__ == '__main__':
    covering_sets = main(m1,S1,P1,uni)

4066 4066 True
1697 4066
2175 4066
2466 4066
2600 4066
2713 4066
2796 4066
2860 4066
2920 4066
2957 4066
2993 4066
3021 4066
3044 4066
3066 4066
3085 4066
3103 4066
3119 4066
3133 4066
3147 4066
3161 4066
3174 4066
3187 4066
3200 4066
3211 4066
3222 4066
3232 4066
3242 4066
3252 4066
3261 4066
3270 4066
3279 4066
3288 4066
3296 4066
3304 4066
3311 4066
3318 4066
3324 4066
3330 4066
3336 4066
3342 4066
3348 4066
3354 4066
3360 4066
3366 4066
3371 4066
3376 4066
3381 4066
3386 4066
3391 4066
3396 4066
3401 4066
3406 4066
3411 4066
3416 4066
3421 4066
3425 4066
3429 4066
3433 4066
3437 4066
3441 4066
3445 4066
3449 4066
3453 4066
3457 4066
3461 4066
3465 4066
3469 4066
3473 4066
3476 4066
3479 4066
3482 4066
3485 4066
3488 4066
3491 4066
3494 4066
3497 4066
3500 4066
3503 4066
3506 4066
3509 4066
3512 4066
3515 4066
3518 4066
3521 4066
3524 4066
3527 4066
3530 4066
3533 4066
3536 4066
3539 4066
3542 4066
3545 4066
3548 4066
3551 4066
3553 4066
3555 4066
3557 4066
3559 4066
3561 4066
3563 

In [153]:
len(covering_sets)

534

In [154]:
covering_roots = []

for cov_set in covering_sets:
    for k in subsets2.keys():
        subset = sorted(subsets2[k])
        if subset == sorted(cov_set):
            covering_roots.append(k)

In [155]:
len(covering_roots)

536

In [157]:
covering_roots

['aeg',
 'mina',
 'aasta',
 'sõna',
 'tema',
 'see',
 'sina',
 'mis',
 'ise',
 'kodu',
 'hommik',
 'õhtu',
 'hetk',
 'lava',
 'juht',
 'algus',
 'teema',
 'vesi',
 'nurk',
 'tänav',
 'keel',
 'selg',
 'teine',
 'silm',
 'Tallinn',
 'pea',
 'käsi',
 'päev',
 'minut',
 'nägu',
 'põhi',
 'suvi',
 'voodi',
 'määr',
 'suu',
 'kes',
 'maailm',
 'tee',
 'kool',
 'august',
 'ulatus',
 'aken',
 'tasku',
 'maa',
 'mood',
 'saal',
 'alus',
 'tase',
 'rida',
 'õu',
 'arvuti',
 'internet',
 'kõrv',
 'ahi',
 'Eesti',
 'töö',
 'september',
 'auto',
 'öö',
 'ühiskond',
 'aed',
 'baar',
 'stiil',
 'tool',
 'laut',
 'rõõm',
 'plikatartu',
 'lõpp',
 'hinnang',
 'koht',
 'inimene',
 'laupäev',
 'esmaspäev',
 'maja',
 'aprill',
 'november',
 'küsimus',
 'USA',
 'suund',
 'kuju',
 'pind',
 'süda',
 'tagajärg',
 'finaal',
 'põrand',
 'tuul',
 'uni',
 'pudel',
 'priva',
 'pää',
 'sää',
 'päike',
 'laup',
 'andmed',
 'mõte',
 'kord',
 'riik',
 'väide',
 'Tartu',
 'sügis',
 'jaanuar',
 'pool',
 'elu',
 'oktoobe

In [160]:
with open('verb_analysis_kohakaanded_root_mustrid_set_cover_greedy_1_roots.txt', 'w') as f:
    for line in covering_roots:
        f.write(f"{line}\n")

In [159]:
import csv

In [161]:
with open("verb_analysis_kohakaanded_root_mustrid_set_cover_greedy_1_sets.csv", "w") as f:
    wr = csv.writer(f)
    wr.writerows(covering_sets)

In [163]:
import pickle 

In [164]:


with open('set_cover_1_verb2num.pkl', 'wb') as f:
    pickle.dump(verb_to_num, f)
        

with open('set_cover_1_num2verb.pkl', 'wb') as f:
    pickle.dump(num_to_verb, f)

In [165]:
with open('set_cover_1_verb2num.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)

print(len(verb_to_num))
print(len(loaded_dict))

4066
4066


In [ ]:
#m1= 5 # maksimaalne väärtus üle setide??
#S1 = [[1,3],[2],[1,2,5],[3,5],[4],[5],[1,3],[2,4,5],[1,2],[2,3]] # setid
#P1 = [11,4,9,12,5,4,13,12,8,9] # mille alusel cost?